<a href="https://github.com/N3iKos/segsmaker-fast">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-segsmaker--fast-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a>

In [ ]:
# @title <b><font color='orange'>⚙️ WebUI Installer</font></b> {"display-mode":"form"}

Webui        = 'Forge-Neo'  # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai___Key = ''           # @param {type:"string", placeholder:"Civitai API Key — required. Get it at civitai.com/user/account"}
HF_Read_Token = ''           # @param {type:"string", placeholder:"HuggingFace READ Token — optional"}
Mount__GDrive = 'No'         # @param ["Yes", "No"]

# ── Google Drive mount (Colab only) ──────────────────────────────────────
import sys
_IS_COLAB = 'google.colab' in sys.modules or 'COLAB_BACKEND_VERSION' in __import__('os').environ

mount = Mount__GDrive
if mount == 'Yes':
    if _IS_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('⚠️  Google Drive mount is only available in Google Colab. Skipping.')
        mount = 'No'

# ── Bootstrap: download and run setup.py ─────────────────────────────────
!curl -sLo /content/setup.py https://github.com/N3iKos/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

# ── Google Drive symlinks (after WebUI is installed) ─────────────────────
if mount == 'Yes':
    from pathlib import Path

    _gdrive_root = Path('/content/drive/MyDrive/segsmaker-fast')
    _gdrive_root.mkdir(parents=True, exist_ok=True)

    for _name, _path in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        _folder = _gdrive_root / _name
        _folder.mkdir(parents=True, exist_ok=True)
        _sym = _path / f'drive-{_name}'
        if not _sym.exists():
            _sym.symlink_to(_folder, target_is_directory=True)

    !rm -rf $WebUI_Output
    _out_name = {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    _out_folder = _gdrive_root / _out_name
    _out_folder.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(_out_folder, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        _wc = WebUI / 'cache'
        !rm -rf $_wc
        _cache_folder = _gdrive_root / 'cache'
        _cache_folder.mkdir(parents=True, exist_ok=True)
        _wc.symlink_to(_cache_folder, target_is_directory=True)


In [ ]:
# @title <b><font color='cyan'>📦 Model Downloader</font></b> {"display-mode":"form"}

# ── Download Options ─────────────────────────────────────────────────────
Parallel_Download = False  # @param {type:"boolean"}
Max_Workers       = 3      # @param {type:"slider", min:1, max:8, step:1}
Load_from_Drive   = False  # @param {type:"boolean"}

# ── Checkpoint ───────────────────────────────────────────────────────────
# Enter one URL per line. Supports: HuggingFace, Civitai, direct links.
Checkpoint_URL = ''  # @param {type:"string", placeholder:"https://civitai.com/... or https://huggingface.co/..."}

# ── LoRA ─────────────────────────────────────────────────────────────────
LoRA_URL = ''  # @param {type:"string", placeholder:"https://civitai.com/... or https://huggingface.co/..."}

# ── VAE ──────────────────────────────────────────────────────────────────
VAE_URL = ''  # @param {type:"string", placeholder:"https://huggingface.co/..."}

# ── Download Logic ───────────────────────────────────────────────────────
import sys, os

def _parse_urls(raw):
    return [u.strip() for u in raw.replace(',', '\n').splitlines() if u.strip()]

def _gdrive_has(folder_path, url):
    """Check if a file with the same basename already exists in Drive folder."""
    from pathlib import Path
    name = url.rstrip('/').split('/')[-1].split('?')[0]
    drive_sym = Path(str(folder_path)).parent / f'drive-{Path(str(folder_path)).name}'
    return (drive_sym / name).exists() if drive_sym.exists() else False

def _run_downloads(url_list, dest_var, parallel, workers):
    from pathlib import Path
    import subprocess, threading, time

    dest = Path(str(dest_var))

    if parallel and len(url_list) > 1:
        # ── Parallel via aria2c ThreadPool ──────────────────────────────
        from concurrent.futures import ThreadPoolExecutor, as_completed
        COLORS = ['\033[91m', '\033[93m', '\033[92m', '\033[96m',
                  '\033[94m', '\033[95m', '\033[97m', '\033[90m']
        RST = '\033[0m'
        lock = threading.Lock()

        def dl_one(idx_url):
            idx, url = idx_url
            color = COLORS[idx % len(COLORS)]
            name = url.rstrip('/').split('/')[-1].split('?')[0] or f'file_{idx}'
            print(f'{color}[Worker-{idx+1}]{RST} Starting: {name}')
            result = subprocess.run(
                ['aria2c', '-x16', '-s16', '-k1M', '--console-log-level=warn',
                 '--summary-interval=0', '-d', str(dest), url],
                capture_output=True, text=True
            )
            if result.returncode == 0:
                print(f'{color}[Worker-{idx+1}]{RST} ✅ Done: {name}')
            else:
                print(f'{color}[Worker-{idx+1}]{RST} ❌ Failed: {name}')
                if result.stderr:
                    print(f'{color}    {RST}{result.stderr[:200]}')

        with ThreadPoolExecutor(max_workers=workers) as pool:
            list(pool.map(dl_one, enumerate(url_list)))
    else:
        # ── Sequential via %download magic ──────────────────────────────
        import subprocess
        os.chdir(str(dest))
        for url in url_list:
            get_ipython().run_line_magic('download', url)

# ── Run downloads ─────────────────────────────────────────────────────────
_ckpt_urls = _parse_urls(Checkpoint_URL)
_lora_urls = _parse_urls(LoRA_URL)
_vae_urls  = _parse_urls(VAE_URL)

if Load_from_Drive:
    print('ℹ️  Load_from_Drive enabled — files already synced from Drive will be skipped.')

if _ckpt_urls:
    print(f'\n📥 Checkpoint ({"parallel" if Parallel_Download else "sequential"})...')
    _run_downloads(_ckpt_urls, CKPT, Parallel_Download, Max_Workers)

if _lora_urls:
    print(f'\n📥 LoRA ({"parallel" if Parallel_Download else "sequential"})...')
    _run_downloads(_lora_urls, LORA, Parallel_Download, Max_Workers)

if _vae_urls:
    print(f'\n📥 VAE ({"parallel" if Parallel_Download else "sequential"})...')
    _run_downloads(_vae_urls, VAE, Parallel_Download, Max_Workers)

if not any([_ckpt_urls, _lora_urls, _vae_urls]):
    print('ℹ️  No URLs provided. Add URLs to the fields above and re-run.')


In [ ]:
# @title <b><font color='cyan'>🧩 Extra Assets</font></b> {"display-mode":"form"}

# ── Download Options ─────────────────────────────────────────────────────
Parallel_Download = False  # @param {type:"boolean"}
Max_Workers       = 3      # @param {type:"slider", min:1, max:8, step:1}
Load_from_Drive   = False  # @param {type:"boolean"}

# ── Extensions / Custom Nodes ─────────────────────────────────────────────
# GitHub repository URL(s) to clone — one per line.
Extension_URL = ''  # @param {type:"string", placeholder:"https://github.com/author/extension-name"}

# ── Embeddings ────────────────────────────────────────────────────────────
Embedding_URL = ''  # @param {type:"string", placeholder:"https://civitai.com/... or https://huggingface.co/..."}

# ── Upscalers ─────────────────────────────────────────────────────────────
Upscaler_URL = ''  # @param {type:"string", placeholder:"https://huggingface.co/..."}

# ── Clone Extensions / Custom Nodes ──────────────────────────────────────
_ext_urls = [u.strip() for u in Extension_URL.replace(',', '\n').splitlines() if u.strip()]
if _ext_urls:
    import os
    _ext_dir = str(Extensions) if 'Extensions' in dir() else str(WebUI / ('custom_nodes' if Webui == 'ComfyUI' else 'extensions'))
    os.chdir(_ext_dir)
    print(f'\n🔗 Cloning {len(_ext_urls)} extension(s)...')
    for _u in _ext_urls:
        get_ipython().run_line_magic('clone', _u)

# ── Download helper (reuse from Cell 2 if already run) ───────────────────
import os

def _parse_urls(raw):
    return [u.strip() for u in raw.replace(',', '\n').splitlines() if u.strip()]

def _run_downloads(url_list, dest_var, parallel, workers):
    from pathlib import Path
    from concurrent.futures import ThreadPoolExecutor
    import subprocess, threading

    dest = Path(str(dest_var))
    COLORS = ['\033[91m','\033[93m','\033[92m','\033[96m','\033[94m','\033[95m','\033[97m','\033[90m']
    RST = '\033[0m'

    if parallel and len(url_list) > 1:
        def dl_one(idx_url):
            idx, url = idx_url
            color = COLORS[idx % len(COLORS)]
            name = url.rstrip('/').split('/')[-1].split('?')[0] or f'file_{idx}'
            print(f'{color}[Worker-{idx+1}]{RST} Starting: {name}')
            r = subprocess.run(['aria2c','-x16','-s16','-k1M','--console-log-level=warn','--summary-interval=0','-d',str(dest),url],capture_output=True,text=True)
            print(f'{color}[Worker-{idx+1}]{RST} {"✅ Done" if r.returncode==0 else "❌ Failed"}: {name}')
        with ThreadPoolExecutor(max_workers=workers) as pool:
            list(pool.map(dl_one, enumerate(url_list)))
    else:
        os.chdir(str(dest))
        for url in url_list:
            get_ipython().run_line_magic('download', url)

_emb_urls = _parse_urls(Embedding_URL)
_ups_urls = _parse_urls(Upscaler_URL)

if _emb_urls:
    print(f'\n📥 Embeddings ({"parallel" if Parallel_Download else "sequential"})...')
    _run_downloads(_emb_urls, Embeddings, Parallel_Download, Max_Workers)

if _ups_urls:
    print(f'\n📥 Upscalers ({"parallel" if Parallel_Download else "sequential"})...')
    _ups_dest = Upscalers if 'Upscalers' in dir() else WebUI / 'models/ESRGAN'
    _run_downloads(_ups_urls, _ups_dest, Parallel_Download, Max_Workers)

if not any([_ext_urls, _emb_urls, _ups_urls]):
    print('ℹ️  No URLs provided. Add URLs above and re-run.')


In [ ]:
# @title <b><font color='cyan'>⚡ FLUX Model Downloader</font></b> {"display-mode":"form"}

# ── FLUX Download Options ─────────────────────────────────────────────────
Parallel_Download = False   # @param {type:"boolean"}
Max_Workers       = 3       # @param {type:"slider", min:1, max:8, step:1}
Load_from_Drive   = False   # @param {type:"boolean"}

# ── FLUX Variant ─────────────────────────────────────────────────────────
FLUX_Variant = 'flux1-dev'  # @param ["flux1-dev", "flux1-schnell", "custom"]

# ── Custom URLs (used only when FLUX_Variant == 'custom') ────────────────
FLUX_Unet_URL = ''  # @param {type:"string", placeholder:"FLUX Unet .safetensors URL"}
FLUX_ClipL_URL = '' # @param {type:"string", placeholder:"FLUX Clip-L .safetensors URL"}
FLUX_T5XXL_URL = '' # @param {type:"string", placeholder:"FLUX T5-XXL .safetensors URL"}
FLUX_VAE_URL   = '' # @param {type:"string", placeholder:"FLUX VAE .safetensors URL"}

# ── WebUI FLUX Compatibility Check ────────────────────────────────────────
import os, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import threading

_FLUX_SUPPORTED_UI = {'Forge', 'Forge-Neo', 'ComfyUI', 'SwarmUI'}
_webui_name = Webui if 'Webui' in dir() else ''

if _webui_name not in _FLUX_SUPPORTED_UI:
    print(f'⚠️  FLUX is not supported for {_webui_name or "the selected WebUI"}.')
    print(f'   FLUX support requires one of: {", ".join(sorted(_FLUX_SUPPORTED_UI))}')
else:
    # ── Preset URLs ───────────────────────────────────────────────────────
    _FLUX_PRESETS = {
        'flux1-dev': {
            'unet':  'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors',
            'clipL': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
            't5xxl': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors',
            'vae':   'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors',
        },
        'flux1-schnell': {
            'unet':  'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors',
            'clipL': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
            't5xxl': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors',
            'vae':   'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors',
        },
    }

    if FLUX_Variant == 'custom':
        _flux_urls = {
            'unet':  FLUX_Unet_URL.strip(),
            'clipL': FLUX_ClipL_URL.strip(),
            't5xxl': FLUX_T5XXL_URL.strip(),
            'vae':   FLUX_VAE_URL.strip(),
        }
    else:
        _flux_urls = _FLUX_PRESETS.get(FLUX_Variant, {})

    # Resolve destination paths from environment variables set by setup.py
    _flux_dest = {
        'unet':  Path(os.environ.get('UNET', str(WebUI / 'models/unet'))),
        'clipL': Path(os.environ.get('CLIP', str(WebUI / 'models/text_encoder'))),
        't5xxl': Path(os.environ.get('CLIP', str(WebUI / 'models/text_encoder'))),
        'vae':   Path(os.environ.get('VAE',  str(WebUI / 'models/VAE'))),
    }

    COLORS = ['\033[91m','\033[93m','\033[92m','\033[96m']
    RST = '\033[0m'

    def _dl_flux_component(idx_item):
        idx, (comp, url) = idx_item
        if not url:
            return
        color = COLORS[idx % len(COLORS)]
        dest = _flux_dest[comp]
        dest.mkdir(parents=True, exist_ok=True)
        name = url.rstrip('/').split('/')[-1].split('?')[0]

        if Load_from_Drive:
            _drive_check = Path('/content/drive/MyDrive/segsmaker-fast') / comp / name
            if _drive_check.exists():
                print(f'{color}[{comp}]{RST} ✅ Already in Drive, skipping: {name}')
                return

        print(f'{color}[{comp}]{RST} Downloading: {name}')
        r = subprocess.run(
            ['aria2c','-x16','-s16','-k1M','--console-log-level=warn','--summary-interval=0','-d',str(dest),url],
            capture_output=True, text=True
        )
        print(f'{color}[{comp}]{RST} {"✅ Done" if r.returncode==0 else "❌ Failed"}: {name}')

    _active = [(k, v) for k, v in _flux_urls.items() if v]
    if not _active:
        print('ℹ️  No FLUX URLs configured. Select a preset or fill custom URLs.')
    else:
        print(f'⚡ Downloading FLUX components for {_webui_name} ({"parallel" if Parallel_Download else "sequential"})...')
        if Parallel_Download and len(_active) > 1:
            with ThreadPoolExecutor(max_workers=min(Max_Workers, len(_active))) as pool:
                list(pool.map(_dl_flux_component, enumerate(_active)))
        else:
            for item in enumerate(_active):
                _dl_flux_component(item)
        print('\n✅ FLUX download complete.')


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget

In [ ]:
# @title <b><font color='orange'>🚀 Launcher</font></b> {"display-mode":"form"}

# ── Launch Arguments ─────────────────────────────────────────────────────
Launch_Args = '--xformers'  # @param {type:"string", placeholder:"e.g. --xformers  or  --disable-xformers --opt-sdp-attention"}

# ── Advanced Flags (hidden features) ─────────────────────────────────────
Skip_Widget     = False  # @param {type:"boolean"}
Skip_ComfyUI_Check = False  # @param {type:"boolean"}

# ── Tunnel ───────────────────────────────────────────────────────────────
Tunnel = 'Gradio'  # @param ["Gradio", "Pinggy", "Cloudflared", "NGROK", "ZROK"]
NGROK_Token = ''   # @param {type:"string", placeholder:"NGROK auth token (only if Tunnel = NGROK)"}
ZROK_Token  = ''   # @param {type:"string", placeholder:"ZROK token (only if Tunnel = ZROK)"}

# ── Build command ─────────────────────────────────────────────────────────
import os
os.chdir(str(WebUI))

_cmd_parts = ['%run segsmaker.py']
if Launch_Args.strip():     _cmd_parts.append(Launch_Args.strip())
if Skip_Widget:             _cmd_parts.append('--skip-widget')
if Skip_ComfyUI_Check:      _cmd_parts.append('--skip-comfyui-check')
if Tunnel == 'NGROK' and NGROK_Token.strip():  _cmd_parts.extend(['--N', NGROK_Token.strip()])
if Tunnel == 'ZROK'  and ZROK_Token.strip():   _cmd_parts.extend(['--Z', ZROK_Token.strip()])

print(f'🚀 Launching: {" ".join(_cmd_parts)}')
get_ipython().run_line_magic('run', ' '.join(['segsmaker.py'] + _cmd_parts[1:]))


## 🛠️ Extras
Utility cells — run individually as needed.

In [ ]:
# @title Storage Info
%storage

In [ ]:
# @title Clear Output Images
%clear_output_images

In [ ]:
# @title Uninstall WebUI
%uninstall_webui

In [ ]:
# @title Zip Output Images
%%zipping

name    = 'my_outputs'
inputs  = $WebUI_Output
outputs = $HOME


In [ ]:
# @title Change Civitai API Key
%change_key

In [ ]:
# @title Register ZROK Account
%zrok_register